# Part 2 — 전처리·학습 데이터 구축

**목적** 원본 JSON을 모델 입력 형식으로 바꾸고, train/dev와 Stage별 subset을 고정한다.

**실행한 것** `python scripts/build_dataset.py` (약 3분) → `data/processed/`, `data/subsets/`, `artifacts/preprocess/<ts>_build_dataset/`

**핵심 설계**
1. jsonl에는 **원문 그대로의 발화 목록 + 원본 요약문**만 저장한다
2. 모델 입력 문자열(`source`)은 학습 때 `configs/base.yaml`의 `preprocess` 설정으로 `src.preprocess.build_source`가 만든다
3. 그래서 입력 버전 6가지(직렬화 3 × 시스템 태그 2)를 파일 하나로 비교할 수 있다

In [1]:
import sys, json, yaml
from pathlib import Path
sys.path.insert(0, "..")
import pandas as pd
from src.preprocess import normalize, build_source, selftest
from src.data_loader import read_jsonl

ROOT = Path("..")
pcfg = yaml.safe_load((ROOT / "configs/base.yaml").read_text(encoding="utf-8"))["preprocess"]
RUN = sorted((ROOT / "artifacts/preprocess").glob("*_build_dataset"))[-1]
metrics = json.loads((RUN / "metrics.json").read_text(encoding="utf-8"))
print("run:", RUN.name)
pcfg

run: 20260919_134736_build_dataset


{'serialization': 'speaker',
 'system_tag': 'drop',
 'emoticon_tag': 'drop',
 'collapse_repeat_jamo': True,
 'collapse_repeat_punct': True,
 'prefix': '대화 요약: '}

## 1. 정규화 규칙 — 입력에만 적용한다

정답 요약문은 절대 건드리지 않는다. 정답을 바꾸면 평가가 무의미해진다.

Part 1에서 `#@시스템#`이 **항상** `#@시스템#사진#`처럼 하위 유형과 함께 나온다는 걸 확인했다.
그래서 태그 규칙은 하위 유형까지 한 덩어리로 처리하고, 바로 뒤에 붙은 다른 태그(`#@이름#`)의 `#`은 건드리지 않는다.

In [2]:
drop = {**pcfg, "system_tag": "drop"}
typ = {**pcfg, "system_tag": "type"}
examples = [
    "이거 봐#@시스템#사진#ㅋㅋㅋㅋㅋ",
    "#@시스템#검색##@이름#야",
    "너무해#@이모티콘#삐짐#너무해#@이모티콘#삐짐#",
    "#@이모티콘#ㅋㅋ#@이름#",
    "진짜!!!!??? 3.5킬로 ......",
    "#@이름# 안녕",
]
pd.DataFrame([{"원문": t, "system_tag=drop": normalize(t, drop), "system_tag=type": normalize(t, typ)} for t in examples])

,원문,system_tag=drop,system_tag=type
0,이거 봐#@시스템#사진#ㅋㅋㅋㅋㅋ,이거 봐 ㅋㅋ,이거 봐 [사진] ㅋㅋ
1,#@시스템#검색##@이름#야,#@이름#야,[검색] #@이름#야
2,너무해#@이모티콘#삐짐#너무해#@이모티콘#삐짐#,너무해 너무해,너무해 너무해
3,#@이모티콘#ㅋㅋ#@이름#,ㅋㅋ#@이름#,ㅋㅋ#@이름#
4,진짜!!!!??? 3.5킬로 ......,진짜!!?? 3.5킬로 ..,진짜!!?? 3.5킬로 ..
5,#@이름# 안녕,#@이름# 안녕,#@이름# 안녕


In [3]:
# 경계 사례를 포함한 자체 테스트 (scripts/build_dataset.py 도 시작할 때 이걸 먼저 돌린다)
selftest()

selftest: 19/19 통과


True

### train 전수 점검 — 규칙이 잔해를 남기지 않는가

- `stray_hash_*`: 알려진 태그를 뺀 뒤에도 남는 `#`의 개수. 정규화 전 44,201개는 대부분 `#@시스템#사진#`의 `사진#` 부분이다
- 정규화 후 195개는 사용자가 원래 입력한 해시태그·오타(`#축하`, `괜찮네#`)이고, 규칙이 새로 만든 잔해는 0개다
- `dialogues_emptied` 0: 태그를 지운 뒤 발화가 하나도 남지 않은 대화는 없다

In [4]:
pd.DataFrame(metrics['normalization']).T

,stray_hash_raw,stray_hash_after,system_tag_left,emoticon_tag_left,utterances_emptied,dialogues_emptied
drop,44201,195,0,0,60523,0
type,44201,195,0,0,18940,0


## 2. 직렬화 3종 — 대화를 한 줄 텍스트로 펴는 방법

| 이름 | 형태 | 특징 |
|---|---|---|
| `plain` | 발화만 개행 연결 | 가장 짧다. 누가 말했는지 모른다 |
| `speaker` | `P01: 발화` | 화자가 보인다. 발화마다 약 5토큰을 더 쓴다 (기본값) |
| `turn` | 같은 턴의 연속 발화를 한 줄로 합침 | 화자 전환이 또렷하고 `speaker`보다 18% 짧다 |

실제 train 샘플 하나를 6가지 버전으로 만들어 본다. `#@시스템#사진#`이 들어 있는 대화다.

In [5]:
rec = next(r for r in read_jsonl(ROOT / "data/subsets/stage_a.jsonl")
           if any("#@시스템#사진#" in u["text"] for u in r["dialogue"]) and len(r["dialogue"]) <= 10)
print("[원문 발화]")
for u in rec["dialogue"]:
    print(f"  {u['pid']}({u['turn']}): {u['text']}")
print()
print("[정답]", rec["target"])

[원문 발화]
  P01(T1): #@시스템#사진#
  P01(T1): 오떙
  P02(T2): 오오오오
  P02(T2): 이쁘다
  P02(T2): 완전 이쁘다
  P01(T3): ㅋㅋㅋㅋ후훗
  P01(T3): 쫌 귀여워졋지
  P02(T4): 완전!

[정답] 사진을 보고 서로 이쁘다고 하니 귀여워졌다 말한다.


In [6]:
for ser in ["plain", "speaker", "turn"]:
    for sysm in ["drop", "type"]:
        print(f"===== {ser} / {sysm}")
        print(build_source(rec, {**pcfg, "serialization": ser, "system_tag": sysm}))
        print()

===== plain / drop
대화 요약: 오떙
오오오오
이쁘다
완전 이쁘다
ㅋㅋ후훗
쫌 귀여워졋지
완전!

===== plain / type
대화 요약: [사진]
오떙
오오오오
이쁘다
완전 이쁘다
ㅋㅋ후훗
쫌 귀여워졋지
완전!

===== speaker / drop
대화 요약: P01: 오떙
P02: 오오오오
P02: 이쁘다
P02: 완전 이쁘다
P01: ㅋㅋ후훗
P01: 쫌 귀여워졋지
P02: 완전!

===== speaker / type
대화 요약: P01: [사진]
P01: 오떙
P02: 오오오오
P02: 이쁘다
P02: 완전 이쁘다
P01: ㅋㅋ후훗
P01: 쫌 귀여워졋지
P02: 완전!

===== turn / drop
대화 요약: P01: 오떙
P02: 오오오오 이쁘다 완전 이쁘다
P01: ㅋㅋ후훗 쫌 귀여워졋지
P02: 완전!

===== turn / type
대화 요약: P01: [사진] 오떙
P02: 오오오오 이쁘다 완전 이쁘다
P01: ㅋㅋ후훗 쫌 귀여워졋지
P02: 완전!



## 3. 정규화 후 입력 길이 — 한도 384 안에 들어가는가

Part 1(정규화 전) `speaker` p99는 351이었다. 정규화 후 325로 줄었지만 256보다는 여전히 커서 `max_source_length` 384와 batch 4를 유지한다.

In [7]:
pd.DataFrame(metrics["source_tokens"]).T[["mean", "p50", "p95", "p99", "max", "fit_256_pct", "fit_384_pct"]]

,mean,p50,p95,p99,max,fit_256_pct,fit_384_pct
plain/drop,93.7,85.0,165.0,232.0,1514.0,99.42,99.94
plain/type,94.3,86.0,166.0,232.0,1514.0,99.41,99.94
speaker/drop,149.0,138.0,243.0,325.0,1584.0,96.21,99.66
speaker/type,150.3,140.0,245.0,327.0,1584.0,96.08,99.65
turn/drop,122.8,114.0,202.0,274.0,1529.0,98.55,99.86
turn/type,123.5,114.0,203.0,275.0,1529.0,98.52,99.86


## 4. 분할 — dialogueID 해시로 고정

- 각 대화의 순위를 `sha256("<용도>:<seed>:<dialogueID>")`로 정한다. 파일을 읽는 순서나 컴퓨터가 달라도 같은 분할이 나온다
- dev는 주제 × 유형 18개 층에 비례 배분한다. 그러면 dev의 토론 대화는 8건뿐이라, 토론 전용 `dev_debate` 200건을 따로 뗐다
- stage_a와 stage_b는 같은 salt로 뽑아서 **stage_a ⊂ stage_b**다. Stage A→B로 키울 때 데이터 차이가 "더 많아진 것"뿐이다

In [8]:
pd.Series(metrics['sizes'], name='건수').to_frame()

,건수
train,278292
dev,1500
dev_debate,200
stage_a,2000
stage_b,20000


In [9]:
pd.Series(metrics['checks'], name='결과').to_frame()

,결과
dev_train_overlap,0
debate_train_overlap,0
dev_debate_overlap,0
stage_a_in_stage_b,True
total_preserved,279992


In [10]:
pd.DataFrame({k: {t: v["pct"] for t, v in d.items()} for k, d in metrics["topic"].items()})

,train,dev,stage_a,stage_b
개인 및 관계,25.41,25.40,25.40,25.41
주거와 생활,16.14,16.13,16.15,16.14
여가 생활,12.59,12.60,12.60,12.58
식음료,10.77,10.80,10.75,10.78
상거래(쇼핑),9.39,9.40,9.40,9.38
행사,7.62,7.60,7.60,7.62
일과 직업,7.30,7.27,7.30,7.30
미용과 건강,6.10,6.13,6.10,6.09
시사/교육,4.69,4.67,4.70,4.69


In [11]:
pd.DataFrame({k: {t: v["n"] for t, v in d.items()} for k, d in metrics["type"].items()}).fillna(0).astype(int)

,train,dev,dev_debate,stage_a,stage_b
일상 대화,276892,1492,0,1990,19900
토론 대화,1400,8,200,10,100


## 5. 결정 요약과 다음 단계

| 항목 | 선택 | 되돌릴 조건 |
|---|---|---|
| 데이터 형식 | jsonl에 원문 저장, source는 config로 생성 | Stage C에서 source 생성 시간이 학습의 5%를 넘으면 확정 버전만 캐시 |
| 시스템 태그 | `drop` / `type` 두 버전 유지 | Stage A에서 `type`이 유의하게 나으면 기본값 변경 |
| 이모티콘 태그 | 하위 유형 포함 삭제 | — (이번에는 변수를 늘리지 않는다) |
| 입력 길이 | 384 유지 | `plain` 확정 시 256 + batch 8 |

**Stage A 비교 순서** (한 번에 한 변수): 기준 `speaker/drop` → 직렬화 `plain`·`turn` → 시스템 태그 `type`